In [2]:
import pandas as pd
import numpy as np
import dai

def main(datasources, start_date, end_date):
    """
    高量柱换手联动主力行为综合因子
    输出：date, instrument, factor（+2/+4/-5/0）
    """
    bar1m = datasources["bar1m"]
    LOOKBACK_DAYS = 90
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d %H:%M:%S')

    # ---- 聚合日线数据 ----
    sql = f"""
    WITH daily_bar AS (
        SELECT
            date::DATE AS date,
            instrument,
            FIRST(open)   AS open,
            MAX(high)     AS high,
            MIN(low)      AS low,
            LAST(close)   AS close,
            SUM(volume)   AS volume,
            SUM(amount)   AS amount
        FROM {bar1m}
        WHERE close > 0
        GROUP BY date::DATE, instrument
    )
    SELECT *
    FROM daily_bar
    """
    df = dai.query(sql, filters={'date': [query_start, end_date]}, compression=True).df()
    if df.empty:
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])

    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['instrument', 'date']).reset_index(drop=True)

    # ---- 统一数值类型（关键修复） ----
    numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'amount']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # ---- 分组处理每个股票 ----
    def process_group(group):
        group = group.sort_values('date').reset_index(drop=True)
        n = len(group)
        # 确保 volume 是 float
        group['volume'] = group['volume'].astype(float)

        # 预计算均线
        group['vol_ma5'] = group['volume'].rolling(5, min_periods=1).mean()
        group['vol_ma60'] = group['volume'].rolling(60, min_periods=1).mean()
        group['prev_close'] = group['close'].shift(1)

        # 判定高量柱
        is_high = group['volume'] >= group['vol_ma5'] * 3
        # 跳空高开：open > prev_close (仅当高量柱)
        gap_up = (group['open'] > group['prev_close']) & is_high
        # 支撑线：跳空取前一日low，否则取当日low
        support = np.nan
        support = np.where(is_high, np.where(gap_up, group['low'].shift(1), group['low']), np.nan)
        group['support'] = support
        group['is_high'] = is_high

        # 初始化状态变量
        factor_vals = np.zeros(n, dtype=float)
        high_records = []          # 存储高量信息 (date, support, low, volume)
        last_support = np.nan      # 最近的高量支撑线
        high_date_idx = -1         # 最近高量在group中的索引
        break_since_last = False   # 从最近高量至今是否有跌破（低点<支撑）
        consecutive_break_count = 0 # 连续放量跌破计数

        for i in range(n):
            row = group.iloc[i]
            # 1. 更新高量信息
            if row['is_high']:
                # 记录新高量
                high_records.append({
                    'date': row['date'],
                    'support': row['support'],
                    'low': row['low'],
                    'volume': row['volume']
                })
                # 更新最近高量信息
                last_support = row['support']
                high_date_idx = i
                # 重置跌破标记（因为有了新支撑）
                break_since_last = False
                consecutive_break_count = 0

            # 2. 判定是否跌破支撑（收盘价 < last_support）
            if not pd.isna(last_support):
                if row['close'] < last_support:
                    # 放量判定（成交量 > 5日均量 * 1.2）
                    vol_ma5_val = group['vol_ma5'].iloc[i]
                    if vol_ma5_val > 0 and row['volume'] > vol_ma5_val * 1.2:
                        # 换手脉冲条件（用成交量脉冲替代：vol > 5日均量*2.5）
                        if row['volume'] > vol_ma5_val * 2.5:
                            # 满足C档条件（空头出货预警）
                            factor_vals[i] = -5
                            # 一旦触发C，跳过后续A/B判断（C覆盖）
                            continue
                    # 即使不满足C，只要跌破也记录break_since_last
                    break_since_last = True

            # 3. 判断A档（多头锁仓）
            # 条件：存在至少1根高量，且从最近高量至今未跌破，且无连续两根放量跌破
            # 且近5日成交量稳定（用变异系数<0.3），无脉冲（vol < vol_ma5*2.5）
            if not break_since_last and len(high_records) > 0:
                # 稳定条件：近5日成交量变异系数<0.3，且日均量 > 60日均量*0.5（保证活跃）
                vol_ma5_val = group['vol_ma5'].iloc[i]
                vol_ma60_val = group['vol_ma60'].iloc[i]
                if vol_ma5_val > 0 and vol_ma60_val > 0:
                    # 计算近5日标准差（使用rolling）
                    vol_series = group['volume'].iloc[max(0,i-4):i+1]
                    if len(vol_series) >= 3:
                        vol_std = vol_series.std()
                        vol_mean = vol_series.mean()
                        cv = vol_std / (vol_mean + 1e-8)
                    else:
                        cv = 1.0  # 数据不足，认为不稳定
                    stable = (cv < 0.3) and (vol_mean > vol_ma60_val * 0.5)
                else:
                    stable = False
                # 无连续两根放量跌破：检查最近两天
                if i >= 1:
                    prev_break = (group['close'].iloc[i-1] < last_support) and (group['volume'].iloc[i-1] > group['vol_ma5'].iloc[i-1] * 1.2)
                    cur_break = (row['close'] < last_support) and (row['volume'] > vol_ma5_val * 1.2)
                    consecutive_break = prev_break and cur_break
                else:
                    consecutive_break = False

                if stable and not consecutive_break:
                    # 至少满足A档
                    factor_vals[i] = 2
                    # 4. 进一步判断B档（加速强化）
                    # 条件：A档基础上，至少有2根高量，且后一根高量低点>前一根低点，且后一根高量成交量>=前一根*0.8
                    if len(high_records) >= 2:
                        h1 = high_records[-2]
                        h2 = high_records[-1]
                        if (h2['low'] > h1['low']) and (h2['volume'] >= h1['volume'] * 0.8):
                            factor_vals[i] = 4  # B档优先级高于A

        group['factor'] = factor_vals
        return group

    # 应用分组处理
    df = df.groupby('instrument', group_keys=False).apply(process_group)

    # ---- 过滤测试区间 ----
    start_dt = pd.to_datetime(start_date)
    df = df[df['date'] >= start_dt].copy()

    # ---- 对齐中证1000成分股 ----
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
        compression=True
    ).df()
    stk_pool['date'] = pd.to_datetime(stk_pool['date'])
    stk_pool['instrument'] = stk_pool['instrument'].astype(str)

    df['instrument'] = df['instrument'].astype(str)

    final_df = pd.merge(
        stk_pool,
        df[['date', 'instrument', 'factor']],
        how='left',
        on=['date', 'instrument']
    )

    # ---- 填充缺失（默认0） ----
    final_df['factor'] = final_df['factor'].fillna(0.0)
    final_df['factor'] = final_df['factor'].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return final_df[['date', 'instrument', 'factor']]


if __name__ == '__main__':
    from bigmodule import M
    import structlog
    logger = structlog.get_logger()

    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    factor_data = main(datasources, start_date, end_date)
    logger.info(f"因子形状: {factor_data.shape}")
    print(factor_data.head())
    print(factor_data['factor'].value_counts())

    # 评估（可选）
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )

[2026-07-07 01:53:16] [info     ] 因子形状: (242000, 3)
        date instrument  factor
0 2024-01-02  000006.SZ     0.0
1 2024-01-02  000012.SZ     0.0
2 2024-01-02  000016.SZ     0.0
3 2024-01-02  000019.SZ     0.0
4 2024-01-02  000025.SZ     0.0
factor
 0.0    237694
 2.0      3987
 4.0       224
-5.0        95
Name: count, dtype: int64
[2026-07-07 01:53:16] [warning  ] bigalpha_eval._latest version='v4' (use ._latest for dev only, not for prod)
[2026-07-07 01:53:18] [info     ] bigalpha_eval.v4 开始运行 ..
[2026-07-07 01:53:18] [warning  ] 未传入官方评估窗口 start_date/end_date，回退到数据自身范围（仅建议本地调试时使用）
[2026-07-07 01:53:19] [info     ] 对齐中证1000历史成分后，官方评估窗口: 2024-01-02 至 2024-12-31
[2026-07-07 01:53:19] [info     ] ========== 数据检查 ==========
[2026-07-07 01:53:19] [info     ] 通过：列名检查（date/instrument + 至少 1 个因子列） factor_cols=['factor', 'close', 'volume', 'amount', 'turn', 'change_ratio', 'daily_return', 'momentum_5', 'reversal_5', 'volatility_5', 'total_market_cap', 'float_market_cap', 'pe_ttm', 'pb', 'ps